# Transfer Learning

**Transfer learning** reuses a model trained for one task as the starting point for a related task. It is most useful when the new task has little labelled data.

In notebook 01 MobileNetV2 could not classify a bean leaf, because ImageNet has no bean diseases. Its convolutional layers, however, have learnt to detect edges, textures and shapes, and these features are useful for almost any photo. Transfer learning:

- **keeps** the pretrained convolutional base, which turns a photo into features;
- **replaces** the ImageNet classifier on top with a new one for your own classes, and trains only that one at first.

This has three benefits: you reuse what the network learnt from 1.2 million photos, training takes minutes instead of days, and a few hundred images per class are enough.

In this notebook you apply it to photos of bean leaves with two diseases. You build and compare two models:

1. a dense Keras head on features extracted by a frozen pretrained base;
2. the frozen base and a head trained together with data augmentation.

An optional section at the end goes one step further and fine-tunes the top layers of the base.

## Learning Objectives

By the end of this notebook, you should be able to:

- Load an image dataset from folders with `image_dataset_from_directory`.
- Extract features with a frozen pretrained base and train a dense classification head on them.
- Add data augmentation layers to a model built on a frozen pretrained base, and plot augmented images.
- Compare the models on the validation set and evaluate the chosen one once on the test set.
- Optional: fine-tune the top layers of a pretrained base with a low learning rate.

## Setup

In [ ]:
from pathlib import Path

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from keras.applications import mobilenet_v2
from sklearn.metrics import ConfusionMatrixDisplay

keras.utils.set_random_seed(42)
IMAGE_SIZE = (160, 160)  # MobileNetV2 also accepts 160 x 160, which halves the computing time

# One plot style for the whole notebook
plt.rcParams.update(
    {"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3}
)

## 1. The Convolutional Base

A pretrained CNN has two parts: the **convolutional base**, which turns an image into features, and the **classifier** on top, which turns the features into the 1,000 ImageNet classes. For a new task you keep the base, remove the classifier and put a new one for your own classes in its place. There are two ways to train it. The figure is schematic: it draws the base as five blocks, while MobileNetV2 has 17.

<img src="assets/transfer_learning.png" width="1000">

- **Feature extraction (A):** the base stays **frozen**, its weights are not changed, and only the new classifier is trained.
- **Fine-tuning (B):** after that, the top blocks of the base are unfrozen and trained a little as well, with a very low learning rate.

`include_top=False` loads MobileNetV2 without its classifier, and `pooling="avg"` adds a [global average pooling](https://keras.io/api/layers/pooling_layers/global_average_pooling2d/) layer at the end: it averages each of the 1,280 feature maps of the last layer to one number, so the base turns every image into a **feature vector** of 1,280 numbers.

In [ ]:
base = mobilenet_v2.MobileNetV2(
    include_top=False, weights="imagenet", input_shape=IMAGE_SIZE + (3,), pooling="avg"
)
print(f"Convolutional base: {base.count_params():,} parameters, output shape {base.output.shape}")

## 2. Trainable or Not: Freeze the Base

Every layer of a Keras model has a `trainable` attribute. Setting `base.trainable = False` **freezes** the whole base: its weights become non-trainable, so training the new classifier cannot change them. Freezing comes first, because a new classifier starts with random weights and would otherwise send large, random gradients into the pretrained filters and damage them. In the optional fine-tuning section at the end, you unfreeze the top layers on purpose.

In [ ]:
print(f"Before freezing: {len(base.trainable_weights)} trainable and {len(base.non_trainable_weights)} non-trainable weight tensors")
base.trainable = False
print(f"After freezing:  {len(base.trainable_weights)} trainable and {len(base.non_trainable_weights)} non-trainable weight tensors")

## 3. Load the Bean Leaf Data

The [bean disease dataset](https://github.com/AI-Lab-Makerere/ibean) was collected by the Makerere AI Lab in Uganda with smartphones in bean fields. Each photo shows one leaf in one of three classes: **healthy**, **angular leaf spot** and **bean rust**, two diseases that farmers have to recognise early. The repository contains a copy in `data/beans/`, resized to 160 × 160 pixels. The photos are already split into a training, a validation and a test set, and each set has one subfolder per class:

```text
data/beans/
├── train/                     1,034 photos
│   ├── angular_leaf_spot/       345 photos
│   ├── bean_rust/               348 photos
│   └── healthy/                 341 photos
├── validation/                  133 photos
│   ├── angular_leaf_spot/        44 photos
│   ├── bean_rust/                45 photos
│   └── healthy/                  44 photos
└── test/                        128 photos
    ├── angular_leaf_spot/        43 photos
    ├── bean_rust/                43 photos
    └── healthy/                  42 photos
```

The folder name is the label: every photo in `train/healthy/` is a healthy leaf. This is the layout [`image_dataset_from_directory`](https://keras.io/api/data_loading/image/#imagedatasetfromdirectory-function) expects. You give it the folder of one set, for example `data/beans/train`, and it:

- finds the class subfolders and sorts their names alphabetically: `angular_leaf_spot` becomes label 0, `bean_rust` label 1 and `healthy` label 2;
- reads the photos in batches of 32 and resizes them to `image_size` if necessary;
- returns each batch as a pair of images and labels.

The function `load_images` below calls it for one set. Only the training set is shuffled. Without shuffling, the photos would arrive sorted by class, so a batch would hold mostly one class and every training step would push the model towards that class. The validation and test sets are only used to measure the model, where the order does not change the accuracy. Keeping their order means the predictions stay in the same order as the labels, which section 7 needs for the confusion matrix.

In [ ]:
BEANS = Path("data") / "beans"


def load_images(split, shuffle):
    """Return a dataset of (images, labels) batches from one folder: "train", "validation" or "test"."""
    return keras.utils.image_dataset_from_directory(
        BEANS / split, image_size=IMAGE_SIZE, batch_size=32, label_mode="int", shuffle=shuffle, seed=42
    )


train_ds = load_images("train", shuffle=True)
valid_ds = load_images("validation", shuffle=False)
test_ds = load_images("test", shuffle=False)
class_names = train_ds.class_names
print("Classes:", class_names)

In [ ]:
images, labels = next(iter(train_ds))  # the first batch of 32 images
fig, axes = plt.subplots(1, 6, figsize=(16, 3))
for ax, image, label in zip(axes, images, labels):
    ax.imshow(image.numpy().astype("uint8"))
    ax.set_title(class_names[label])
    ax.axis("off")
plt.show()

The classes are balanced, with about the same number of photos each. The two diseases differ in their pattern: angular leaf spot gives small brown spots limited by the leaf veins, bean rust gives many small round rust-coloured spots.

About 1,000 photos are far too few to train a CNN from scratch: its filters start random and would have to learn edges, textures and shapes from these few leaves alone. A pretrained network has already learnt them from 1.2 million ImageNet photos.

## 4. A Dense Head on the Frozen Base

### Extract the Features Once

Because the base is frozen, each image always gives the same feature vector. You can therefore run the base once over every image, store the vectors, and train a classifier on them.

The function `extract_features` loops over the batches of a dataset. For each batch of images and labels it:

1. scales the pixels with MobileNetV2's `preprocess_input`;
2. runs the frozen base, which turns each image into 1,280 numbers;
3. stores these features and the labels of the same batch in two lists.

At the end, [`np.concatenate`](https://numpy.org/doc/stable/reference/generated/numpy.concatenate.html) stacks the batches into one table of features and one array of labels. Because features and labels come from the same batch, row *i* of the features always belongs to label *i*.

For the training set, the cell loads the images again without shuffling, so `X_train` has the same row order every time you run the notebook.

In [ ]:
def extract_features(dataset):
    """Run the frozen base over every batch of a dataset; return the feature vectors and the labels."""
    all_features = []
    all_labels = []
    for images, labels in dataset:  # one batch of up to 32 images at a time
        images = mobilenet_v2.preprocess_input(images)  # scale the pixels from 0 to 255 to -1 to 1
        features = base.predict(images, verbose=0)  # one row of 1,280 numbers per image
        all_features.append(features)
        all_labels.append(labels.numpy())
    return np.concatenate(all_features), np.concatenate(all_labels)  # stack the batches into one table


X_train, y_train = extract_features(load_images("train", shuffle=False))
X_valid, y_valid = extract_features(valid_ds)
X_test, y_test = extract_features(test_ds)
print("Training features:", X_train.shape, " validation features:", X_valid.shape)

### Train the Head

The feature vectors form an ordinary table: one row per image, 1,280 numeric columns. The new classifier, the **head**, is a small Keras model on this table: a dropout layer, as in the regularisation notebooks, and a dense layer with a softmax activation for the three classes. Training it takes seconds, because the base does not run again.

The helper functions below train a model and plot its learning curves, as in the neural network notebooks. The dictionary `results` collects the validation accuracy of every model.

In [ ]:
def compile_and_fit(model, epochs, learning_rate=1e-3, initial_epoch=0):
    """Compile a classifier with Adam and train it on the bean images; return the learning curves."""
    model.compile(
        loss="sparse_categorical_crossentropy",
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        metrics=["accuracy"],
    )
    history = model.fit(
        train_ds, validation_data=valid_ds, epochs=epochs, initial_epoch=initial_epoch, verbose=0
    )
    return pd.DataFrame(history.history, index=history.epoch)


def plot_curves(logs, title):
    """Plot the training and validation loss and accuracy of one run."""
    fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(12, 4))
    logs[["loss", "val_loss"]].plot(ax=ax_loss, xlabel="Epoch", title=f"{title}: loss")
    logs[["accuracy", "val_accuracy"]].plot(ax=ax_acc, xlabel="Epoch", title=f"{title}: accuracy")
    plt.show()


results = {}

In [ ]:
keras.utils.clear_session()
keras.utils.set_random_seed(42)
head = keras.Sequential(
    [
        keras.Input(shape=(X_train.shape[1],)),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(units=3, activation="softmax"),
    ]
)
head.compile(loss="sparse_categorical_crossentropy", optimizer=keras.optimizers.Adam(1e-3), metrics=["accuracy"])
head_history = head.fit(X_train, y_train, validation_data=(X_valid, y_valid), epochs=30, verbose=0)
plot_curves(pd.DataFrame(head_history.history), "Dense head on the features")
results["Features + dense head"] = head.evaluate(X_valid, y_valid, verbose=0)[1]
print(f"Validation accuracy: {results['Features + dense head']:.3f}")

The dense head reaches a validation accuracy of 0.910, 121 of the 133 validation images, after training for a few seconds on the extracted features. The frozen base, trained only on ImageNet, already describes the leaves well enough for a single softmax layer to separate the three classes. The curves show the overfitting pattern of the regularisation topic: the training accuracy keeps rising to about 0.98, while the validation accuracy levels off between 0.90 and 0.92 from about epoch 20.

## 5. Data Augmentation

With about 1,000 training images, the classifier receives every leaf in only one position, size and orientation. **Data augmentation** creates new variants of the training images during training, by random flips, rotations and zooms. A leaf photographed upside down is still the same leaf with the same disease, so the network learns features that do not depend on these details, and overfits less.

Keras provides augmentation as [preprocessing layers](https://keras.io/api/layers/preprocessing_layers/image_augmentation/) such as `RandomFlip`, `RandomRotation` and `RandomZoom`. They change the images only during training; when the model predicts or is evaluated, they pass the images through unchanged. To see their effect, call them with `training=True`.

In [ ]:
augmentation = keras.Sequential(
    [
        keras.layers.RandomFlip("horizontal_and_vertical"),
        keras.layers.RandomRotation(0.15),  # up to 15 % of a full turn in either direction
        keras.layers.RandomZoom(0.15),
    ],
    name="augmentation",
)

leaf = images[:1]  # one image from the first training batch
fig, axes = plt.subplots(1, 6, figsize=(16, 3))
axes[0].imshow(leaf[0].numpy().astype("uint8"))
axes[0].set_title("original")
for ax in axes[1:]:
    ax.imshow(augmentation(leaf, training=True)[0].numpy().astype("uint8"))
    ax.set_title("augmented")
for ax in axes:
    ax.axis("off")
plt.show()

Augmentation has a cost: every epoch shows the base different images, so the features can no longer be computed once. The base has to run on every training image in every epoch, inside the model. The model below is a `keras.Sequential` chain of the augmentation, the scaling of MobileNetV2 (a [`Rescaling`](https://keras.io/api/layers/preprocessing_layers/image_preprocessing/rescaling/) layer that maps 0 to 255 onto -1 to 1, like `preprocess_input`), the frozen base and the same head. The whole base goes into the list like a single layer. Ten epochs take about two minutes.

In [ ]:
keras.utils.clear_session()
keras.utils.set_random_seed(42)
base.trainable = False

transfer_model = keras.Sequential(
    [
        keras.Input(shape=IMAGE_SIZE + (3,)),
        augmentation,
        keras.layers.Rescaling(scale=1 / 127.5, offset=-1),  # the scaling of mobilenet_v2.preprocess_input
        base,  # the frozen convolutional base
        keras.layers.Dropout(0.2),
        keras.layers.Dense(units=3, activation="softmax"),
    ]
)

transfer_logs = compile_and_fit(transfer_model, epochs=10)
plot_curves(transfer_logs, "Frozen base with augmentation")
results["Frozen base + augmentation"] = transfer_model.evaluate(valid_ds, verbose=0)[1]
print(f"Validation accuracy: {results['Frozen base + augmentation']:.3f}")

The frozen base with augmentation reaches 0.872 after 10 epochs, five images fewer than the dense head. For most epochs its training accuracy lies below its validation accuracy, because the training images are randomly altered and dropout is active, while the validation images are not: a pattern you met with dropout in the regularisation topic. Augmentation reduces overfitting, but the model learns more slowly, and each epoch takes about ten seconds instead of a fraction of a second.

## 6. Compare the Models on the Validation Set

In [ ]:
pd.Series(results, name="Validation accuracy").round(3).to_frame()

The dense head on the features is highest with 0.910, five validation images ahead of the frozen base with augmentation (0.872). Augmentation reduces overfitting, but within 10 epochs the model has not caught up with the head, which trained for 30 epochs in a fraction of the time. Keep in mind that one validation image is 0.008, and that the validation accuracy of a single model varies by one or two images between epochs.

## Try It: Features from Another Pretrained Network

The features depend on the pretrained network. Extract the features of the bean images with [EfficientNetB0](https://keras.io/api/applications/efficientnet/) (`include_top=False`, `pooling="avg"`, `IMAGE_SIZE`), which you met in notebook 01, and train the same dense head as in section 4 on them. Its `preprocess_input` leaves the pixels unchanged, because the scaling is part of the network. Compare its **validation** accuracy with the 0.910 of the MobileNetV2 features. The test set is kept for section 7.

In [ ]:
from keras.applications import efficientnet

# TODO: create the EfficientNetB0 base without its classifier, with average pooling and IMAGE_SIZE
efficient_base = ...


def extract_efficient_features(dataset):
    """Run EfficientNetB0 over every batch of a dataset; return the feature vectors and the labels."""
    all_features = []
    all_labels = []
    for images, labels in dataset:
        # TODO: apply efficientnet.preprocess_input and efficient_base to the batch, as in extract_features
        features = ...
        all_features.append(features)
        all_labels.append(labels.numpy())
    return np.concatenate(all_features), np.concatenate(all_labels)


E_train, _ = extract_efficient_features(load_images("train", shuffle=False))
E_valid, _ = extract_efficient_features(valid_ds)
keras.utils.clear_session()
keras.utils.set_random_seed(42)
# TODO: build the head of section 4 for E_train.shape[1] features, train it for 30 epochs and print its validation accuracy
efficient_head = ...
...

## 7. Final Check on the Test Set

The dense head on the features has the highest validation accuracy, 0.910, so it is the chosen model. It is also the cheapest: it trains in seconds instead of minutes. The test set has not been used yet; it is used once, here, for the chosen model. A [confusion matrix](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html) shows which classes it confuses.

In [ ]:
test_accuracy = head.evaluate(X_test, y_test, verbose=0)[1]
print(f"Dense head on the features, test accuracy: {test_accuracy:.3f}")

y_pred = head.predict(X_test, verbose=0).argmax(axis=1)  # the class with the highest probability
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=class_names, colorbar=False, ax=ax)
ax.grid(False)
plt.show()

On the test set the dense head reaches 0.898, 115 of 128 images, slightly below its validation accuracy of 0.910, as expected for a model chosen on the validation set. The confusion matrix shows where the errors are: 8 of the 13 mistakes are between the two diseases, mostly bean rust leaves predicted as angular leaf spot (6). Both diseases produce brown spots on the leaf, and at 160 × 160 pixels the difference between small angular spots and small round rust spots is only a few pixels wide. Four healthy leaves are classified as diseased, three of them as bean rust, and one bean rust leaf as healthy.

## Optional: Fine-Tuning

This section is optional. It continues training the model of section 5, so run it after the rest of the notebook. The test set has already been used in section 7, so compare the fine-tuned model with the others on the validation set only.

The features of MobileNetV2 were learnt on ImageNet, where leaf diseases do not appear. **Fine-tuning** adapts the top of the base to the new task: you unfreeze its last layers and continue training the whole model with a learning rate a hundred times lower than before. The low learning rate changes the pretrained filters only slightly; a large one would overwrite what they learnt on ImageNet in the first few updates.

Two rules make fine-tuning work:

- Fine-tune only **after** the new head has been trained. A random head would send large, random gradients into the pretrained base.
- After changing `trainable`, **compile the model again**; Keras only takes the change into account when it compiles.

### Keep Batch Normalisation Frozen

MobileNetV2 contains [batch normalisation](https://keras.io/api/layers/normalization_layers/batch_normalization/) layers, which rescale their inputs with a mean and a variance stored during the ImageNet training. A batch normalisation layer with `trainable = False` always uses these stored values. Once it is unfrozen, it recomputes them from each batch during training, here from small batches of bean images, which changes the output of the whole base at once and undoes part of what the head has learnt.

The cell below therefore keeps the batch normalisation layers frozen explicitly. (The Keras guide on transfer learning builds the model with the functional API and calls the base with `base(x, training=False)` instead; in Keras 3 this is not enough once the layers are unfrozen.) MobileNetV2 has 155 layers; the cell unfreezes the last 30, except the batch normalisation layers among them, and continues training for five epochs. `initial_epoch=10` continues the epoch count of the first phase, so the learning curves of both phases can be plotted together.

In [ ]:
base.trainable = True
for layer in base.layers[:-30]:  # keep everything except the last 30 layers frozen
    layer.trainable = False
for layer in base.layers:  # batch normalisation layers stay frozen
    if isinstance(layer, keras.layers.BatchNormalization):
        layer.trainable = False
print(f"Trainable layers in the base: {sum(layer.trainable for layer in base.layers)} of {len(base.layers)}")

fine_logs = compile_and_fit(transfer_model, epochs=15, learning_rate=1e-5, initial_epoch=10)
plot_curves(pd.concat([transfer_logs, fine_logs]), "Frozen base, then fine-tuning from epoch 10")
results["Fine-tuned"] = transfer_model.evaluate(valid_ds, verbose=0)[1]
print(f"Validation accuracy: {results['Fine-tuned']:.3f}")

Fine-tuning continues smoothly from epoch 10: the training loss does not jump when the top layers are unfrozen, and both losses fall again. The training accuracy rises from about 0.84 to 0.89, and the validation accuracy from 0.872 to 0.932, its highest value, reached in the last epoch. Adapting the top layers to bean leaves adds eight correctly classified validation images in five epochs, and puts the fine-tuned model three images ahead of the dense head (0.910). The gain costs several minutes of training, and three validation images are close to the variation between epochs.

### Feature Extraction or Fine-Tuning?

Both are transfer learning; they differ in how much of the base is trained.

| | Feature extraction | Fine-tuning |
|---|---|---|
| Base | frozen completely | top layers unfrozen, the rest frozen |
| Trained | only the new head | the head first, then the head and the top layers together |
| Learning rate | normal, for example 0.001 | very low, for example 0.00001, so the pretrained filters change only slightly |
| Training time | seconds with precomputed features | minutes: the base runs on every image in every epoch |
| Risk | the features may not fit the new task perfectly | overfitting, and destroying the pretrained features with a high learning rate |
| When to use | small datasets, a first baseline | more data, or images that differ from ImageNet |

## Summary

- Transfer learning reuses the convolutional base of a pretrained network and trains a new classifier for your own classes.
- With a frozen base you can extract the features once and train a small head on them in seconds. Collect the features and labels from the same batches, so they stay in the same order.
- Data augmentation creates new variants of the training images with random flips, rotations and zooms. It requires the base to run inside the model in every epoch.
- Optionally, fine-tuning unfreezes the top layers of the base after the head has been trained, with a learning rate about a hundred times lower; the batch normalisation layers stay frozen.

Next, apply the workflow in [03 - Exercise: Classify Sea Animals with Transfer Learning](03_exercise_sea_animals.ipynb). The optional [04 - Hyperparameter Tuning with Keras Tuner](04_optional_keras_tuner.ipynb) shows how to search the hyperparameters of the head.

## References

- Chollet, F. and Watson, M. (2025). *Deep Learning with Python*, 3rd edition, chapter 8: Image classification, on feature extraction, data augmentation and fine-tuning. [Free to read online](https://deeplearningwithpython.io/).
- Keras guide: [Transfer learning and fine-tuning](https://keras.io/guides/transfer_learning/), including why batch normalisation layers stay frozen.
- TensorFlow tutorial: [Data augmentation](https://www.tensorflow.org/tutorials/images/data_augmentation).
- Makerere AI Lab (2020). [Bean disease dataset](https://github.com/AI-Lab-Makerere/ibean), MIT licence.
- Keras API: [`image_dataset_from_directory`](https://keras.io/api/data_loading/image/#imagedatasetfromdirectory-function), [image augmentation layers](https://keras.io/api/layers/preprocessing_layers/image_augmentation/) and [`MobileNetV2`](https://keras.io/api/applications/mobilenet/).
- The structure follows the transfer learning notebooks of the Spiced Academy deep learning course.